# Analysis Definitions Editor

Interactive editor for `config/analysis_definitions.json` — the catalogue of SQL-templated analyses consumed by `scripts/run_analysis.py`, `scripts/run_experiment_report.py`, and the `complete_analysis.ipynb` notebook.

Workflow: select an entry, edit fields, *Validate*, *Save to List* (in-memory), then *Save File* to persist. Backups are written under `config/.backups/` (timestamped) before any disk write.

Dropdowns for `category`, `data_focus`, and `complexity` are populated from the current contents of the file, so the editor stays in sync with whatever vocabulary the project actually uses (rather than enforcing a hard-coded list).

In [ ]:
import sys
from pathlib import Path

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / 'pyproject.toml').exists() and (_p / 'src').exists():
        sys.path.insert(0, str(_p))
        break

import copy
import json
from datetime import datetime

import ipywidgets as widgets
from IPython.display import clear_output, display

from src.utils.path_utils import find_project_root

PROJECT_ROOT = find_project_root()
DEFINITIONS_FILE = PROJECT_ROOT / 'config' / 'analysis_definitions.json'
BACKUP_DIR = PROJECT_ROOT / 'config' / '.backups'

In [ ]:
REQUIRED_FIELDS = (
    'id', 'category', 'subcategory', 'title', 'description',
    'keywords', 'data_focus', 'complexity', 'query_template',
    'params', 'result_type', 'prerequisites',
)


def load_definitions(path: Path) -> list[dict]:
    return json.loads(path.read_text()) if path.exists() else []


def save_definitions(data: list[dict], path: Path) -> Path | None:
    backup = None
    if path.exists():
        BACKUP_DIR.mkdir(parents=True, exist_ok=True)
        backup = BACKUP_DIR / f'{path.stem}_{datetime.now():%Y%m%d_%H%M%S}.json'
        backup.write_text(path.read_text())
    path.write_text(json.dumps(data, indent=2))
    return backup


def unique_values(data: list[dict], field: str) -> list[str]:
    return sorted({d[field] for d in data if d.get(field)})


def filter_definitions(data, category, subcategory, search):
    out = data
    if category != 'All':
        out = [d for d in out if d.get('category') == category]
    if subcategory != 'All':
        out = [d for d in out if d.get('subcategory') == subcategory]
    if search:
        s = search.lower()
        out = [d for d in out
               if s in d.get('id', '').lower()
               or s in d.get('title', '').lower()
               or any(s in kw.lower() for kw in d.get('keywords', []))]
    return out


def validate_definition(defn: dict) -> list[str]:
    errors = []
    for field in REQUIRED_FIELDS:
        if field not in defn:
            errors.append(f"missing field '{field}'")
        elif not defn[field] and field not in ('params', 'keywords'):
            errors.append(f"empty field '{field}'")
    if not isinstance(defn.get('keywords', []), list):
        errors.append("'keywords' must be a list")
    if not isinstance(defn.get('params', []), list):
        errors.append("'params' must be a list")
    if not isinstance(defn.get('prerequisites', {}), dict):
        errors.append("'prerequisites' must be an object")
    if defn.get('result_type') == 'plot' and not isinstance(defn.get('plot_config'), dict):
        errors.append("result_type='plot' requires 'plot_config' object")
    return errors


def blank_definition() -> dict:
    return {
        'id': f'new_analysis_{datetime.now():%Y%m%d_%H%M%S}',
        'category': 'performance_analysis',
        'subcategory': 'uncategorized',
        'title': 'New Analysis',
        'description': '',
        'keywords': [],
        'data_focus': 'evaluation',
        'complexity': 'simple',
        'query_template': 'SELECT * FROM episodes LIMIT 10;',
        'params': [],
        'result_type': 'dataframe',
        'prerequisites': {'min_sample_size': 30, 'requires_evaluation_data': True},
    }

In [ ]:
class DefinitionEditor:
    """ipywidgets UI for browsing and editing analysis definitions."""

    def __init__(self, path: Path):
        self.path = path
        self.data: list[dict] = []
        self.index: int | None = None
        self.original: dict | None = None
        self._build()
        self.load()

    def _build(self):
        self.file_status = widgets.Output()
        self.reload_btn = widgets.Button(description='Reload', button_style='info')
        self.save_btn = widgets.Button(description='Save File', button_style='success')
        self.reload_btn.on_click(lambda _: self.load())
        self.save_btn.on_click(lambda _: self.save())

        self.category_filter = widgets.Dropdown(options=['All'], value='All', description='Category:')
        self.subcategory_filter = widgets.Dropdown(options=['All'], value='All', description='Subcategory:')
        self.search = widgets.Text(placeholder='id, title, or keyword', description='Search:')
        self.count = widgets.HTML('')
        self.selector = widgets.Select(options=[], rows=12, layout=widgets.Layout(width='100%'))
        self.new_btn = widgets.Button(description='New', button_style='primary')
        self.delete_btn = widgets.Button(description='Delete', button_style='danger')

        for w in (self.category_filter, self.subcategory_filter, self.search):
            w.observe(lambda _c: self.refresh_list(), names='value')
        self.selector.observe(lambda _c: self.load_selected(), names='value')
        self.new_btn.on_click(lambda _: self.create_new())
        self.delete_btn.on_click(lambda _: self.delete_selected())

        self.id_in = widgets.Text(description='ID:')
        self.title_in = widgets.Text(description='Title:')
        self.category_in = widgets.Dropdown(options=[], description='Category:')
        self.subcategory_in = widgets.Text(description='Subcategory:')
        self.data_focus_in = widgets.Dropdown(options=[], description='Data focus:')
        self.complexity_in = widgets.Dropdown(options=[], description='Complexity:')
        self.result_type_in = widgets.Dropdown(options=['dataframe', 'plot'], description='Result type:')
        self.description_in = widgets.Textarea(description='Description:', layout=widgets.Layout(width='100%', height='80px'))
        self.keywords_in = widgets.Text(description='Keywords:', placeholder='comma-separated')
        self.query_in = widgets.Textarea(description='Query:', layout=widgets.Layout(width='100%', height='160px'))
        self.plot_config_in = widgets.Textarea(description='Plot config:', placeholder='JSON object (required if result_type=plot)', layout=widgets.Layout(width='100%', height='120px'))
        self.prerequisites_in = widgets.Textarea(description='Prereqs:', placeholder='JSON object', layout=widgets.Layout(width='100%', height='80px'))

        self.validation = widgets.Output()
        self.validate_btn = widgets.Button(description='Validate', button_style='info')
        self.save_list_btn = widgets.Button(description='Save to List', button_style='success')
        self.reset_btn = widgets.Button(description='Reset', button_style='warning')
        self.validate_btn.on_click(lambda _: self.validate_current())
        self.save_list_btn.on_click(lambda _: self.save_to_list())
        self.reset_btn.on_click(lambda _: self.reset_form())

        browser = widgets.VBox([
            widgets.HTML('<h3>Browse</h3>'),
            widgets.HBox([self.category_filter, self.subcategory_filter]),
            self.search, self.count, self.selector,
            widgets.HBox([self.new_btn, self.delete_btn]),
        ])
        editor = widgets.VBox([
            widgets.HTML('<h3>Edit</h3>'),
            self.id_in, self.title_in, self.category_in, self.subcategory_in,
            self.description_in, self.keywords_in,
            self.data_focus_in, self.complexity_in, self.result_type_in,
            self.query_in, self.plot_config_in, self.prerequisites_in,
            self.validation,
            widgets.HBox([self.validate_btn, self.save_list_btn, self.reset_btn]),
        ])
        self.layout = widgets.VBox([
            widgets.HTML(f'<h2>Analysis Definitions Editor</h2><code>{self.path}</code>'),
            widgets.HBox([self.reload_btn, self.save_btn]),
            self.file_status,
            widgets.HTML('<hr>'),
            widgets.HBox([
                widgets.Box([browser], layout=widgets.Layout(width='42%')),
                widgets.Box([editor], layout=widgets.Layout(width='58%')),
            ]),
        ])

    def display(self):
        display(self.layout)

    def load(self):
        self.data = load_definitions(self.path)
        self.refresh_dropdowns()
        self.refresh_list()
        with self.file_status:
            clear_output(wait=True)
            print(f'Loaded {len(self.data)} definitions from {self.path.name}')

    def save(self):
        backup = save_definitions(self.data, self.path)
        with self.file_status:
            clear_output(wait=True)
            if backup:
                print(f'Backup: {backup.relative_to(PROJECT_ROOT)}')
            print(f'Saved {len(self.data)} definitions to {self.path.name}')

    def refresh_dropdowns(self):
        cats = unique_values(self.data, 'category')
        subs = unique_values(self.data, 'subcategory')
        focuses = unique_values(self.data, 'data_focus') or ['evaluation', 'training']
        comps = unique_values(self.data, 'complexity') or ['simple', 'medium', 'complex']
        self.category_filter.options = ['All'] + cats
        self.subcategory_filter.options = ['All'] + subs
        self.category_in.options = cats or ['performance_analysis']
        self.data_focus_in.options = focuses
        self.complexity_in.options = comps

    def refresh_list(self):
        visible = filter_definitions(self.data, self.category_filter.value, self.subcategory_filter.value, self.search.value)
        visible_ids = {id(d) for d in visible}
        self.selector.options = [
            (f"{d['id']} - {d.get('title', '')}", i)
            for i, d in enumerate(self.data) if id(d) in visible_ids
        ]
        self.count.value = f'<i>{len(visible)} of {len(self.data)} definitions</i>'

    def load_selected(self):
        if self.selector.value is None:
            return
        self.index = self.selector.value
        defn = self.data[self.index]
        self.original = copy.deepcopy(defn)
        self.id_in.value = defn.get('id', '')
        self.title_in.value = defn.get('title', '')
        if defn.get('category') in self.category_in.options:
            self.category_in.value = defn['category']
        self.subcategory_in.value = defn.get('subcategory', '')
        self.description_in.value = defn.get('description', '')
        self.keywords_in.value = ', '.join(defn.get('keywords', []))
        if defn.get('data_focus') in self.data_focus_in.options:
            self.data_focus_in.value = defn['data_focus']
        if defn.get('complexity') in self.complexity_in.options:
            self.complexity_in.value = defn['complexity']
        self.result_type_in.value = defn.get('result_type', 'dataframe')
        self.query_in.value = defn.get('query_template', '')
        self.plot_config_in.value = json.dumps(defn['plot_config'], indent=2) if defn.get('plot_config') else ''
        self.prerequisites_in.value = json.dumps(defn.get('prerequisites', {}), indent=2)
        with self.validation:
            clear_output()

    def gather_form(self) -> tuple[dict, list[str]]:
        errors: list[str] = []
        defn = {
            'id': self.id_in.value.strip(),
            'title': self.title_in.value.strip(),
            'category': self.category_in.value,
            'subcategory': self.subcategory_in.value.strip(),
            'description': self.description_in.value.strip(),
            'keywords': [k.strip() for k in self.keywords_in.value.split(',') if k.strip()],
            'data_focus': self.data_focus_in.value,
            'complexity': self.complexity_in.value,
            'result_type': self.result_type_in.value,
            'query_template': self.query_in.value.strip(),
            'params': [],
        }
        if self.plot_config_in.value.strip():
            try:
                defn['plot_config'] = json.loads(self.plot_config_in.value)
            except json.JSONDecodeError as e:
                errors.append(f'plot_config JSON: {e}')
        try:
            defn['prerequisites'] = json.loads(self.prerequisites_in.value) if self.prerequisites_in.value.strip() else {}
        except json.JSONDecodeError as e:
            errors.append(f'prerequisites JSON: {e}')
        return defn, errors

    def validate_current(self):
        with self.validation:
            clear_output()
            defn, errors = self.gather_form()
            errors.extend(validate_definition(defn))
            if errors:
                print('VALIDATION FAILED')
                for e in errors:
                    print(f'  - {e}')
            else:
                print('VALIDATION PASSED')

    def save_to_list(self):
        with self.validation:
            clear_output()
            defn, errors = self.gather_form()
            errors.extend(validate_definition(defn))
            if errors:
                print('Cannot save:')
                for e in errors:
                    print(f'  - {e}')
                return
            duplicate = any(i != self.index and d['id'] == defn['id'] for i, d in enumerate(self.data))
            if duplicate:
                print(f"Cannot save: duplicate id '{defn['id']}'")
                return
            if self.index is None:
                self.data.append(defn)
                self.index = len(self.data) - 1
                print(f"Added '{defn['id']}' (remember to Save File)")
            else:
                self.data[self.index] = defn
                print(f"Updated '{defn['id']}' (remember to Save File)")
            self.refresh_dropdowns()
            self.refresh_list()

    def reset_form(self):
        if self.original is not None:
            self.data[self.index] = copy.deepcopy(self.original)
            self.load_selected()

    def create_new(self):
        self.data.append(blank_definition())
        self.refresh_list()
        self.selector.value = len(self.data) - 1

    def delete_selected(self):
        if self.index is None:
            return
        removed = self.data.pop(self.index)['id']
        self.index = None
        self.original = None
        self.refresh_list()
        with self.validation:
            clear_output()
            print(f"Deleted '{removed}' (remember to Save File)")


editor = DefinitionEditor(DEFINITIONS_FILE)
editor.display()